# Sesion 2 - Metro Madrid: estacionalidad compleja y forecast

Este notebook continua la sesion 1, pero con una serie diaria real de demanda de metro.

Objetivos de la sesion:
1. Hacer EDA rapido para entender dominio y calendario.
2. Limpiar datos y guardar fichero limpio.
3. Visualizar estacionalidades con serie temporal y heatmap.
4. Detectar anomalias y tratar periodo COVID.
5. Hacer forecast con intervalos de confianza y backtesting.

## 1) Librerias y configuracion minima

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)

## 2) Carga, formato de fecha y limpieza inicial

Convertimos fecha, ordenamos y renombramos para trabajar de forma simple.

In [ ]:
raw_path = "../data/raw/demanda_diaria_metro_2015_2024.csv"
df = pd.read_csv(raw_path)

df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.rename(columns={"Fecha": "date", "Demanda": "y"})
df["y"] = pd.to_numeric(df["y"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)
df = df[["date", "y"]]
df.head()

In [ ]:
# Chequeos rapidos de calidad
print("Filas:", len(df))
print("Fecha min:", df["date"].min())
print("Fecha max:", df["date"].max())
print("Nulos en y:", df["y"].isna().sum())
print("Duplicados de fecha:", df["date"].duplicated().sum())

## 3) Guardado de fichero limpio

Guardamos una version limpia para dejar trazabilidad del preprocesado.

In [ ]:
clean_path = "../data/processed/metro_daily_clean.csv"
df.to_csv(clean_path, index=False)
print("Fichero guardado en:", clean_path)

## 4) EDA de dominio y visualizacion temporal

Empezamos con visualizaciones sencillas para entender comportamiento:
- Serie completa.
- Patron por dia de semana.
- Patron mensual.

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["y"], color="tab:blue", linewidth=1)
plt.title("Demanda diaria de Metro Madrid")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.show()

In [ ]:
df_eda = df.copy()
df_eda["weekday"] = df_eda["date"].dt.day_name()
df_eda["month"] = df_eda["date"].dt.month

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="weekday", y="y", order=weekday_order)
plt.title("Distribucion de demanda por dia de semana")
plt.xlabel("Dia")
plt.ylabel("Viajeros")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="month", y="y")
plt.title("Distribucion de demanda por mes")
plt.xlabel("Mes")
plt.ylabel("Viajeros")
plt.show()

## 5) Heatmap para observar estacionalidad

Creamos una matriz Ano x Mes con demanda media diaria y la representamos con un heatmap.

In [ ]:
df_heat = df.copy()
df_heat["year"] = df_heat["date"].dt.year
df_heat["month"] = df_heat["date"].dt.month

heat_table = df_heat.pivot_table(index="year", columns="month", values="y", aggfunc="mean")

plt.figure(figsize=(12, 5))
sns.heatmap(heat_table, cmap="YlGnBu", annot=True, fmt=".0f")
plt.title("Heatmap de demanda media diaria (Ano x Mes)")
plt.xlabel("Mes")
plt.ylabel("Ano")
plt.show()

## 6) Deteccion de anomalias y periodo COVID

Para esta clase queremos reducir el efecto distorsionador de COVID en el modelo.

Estrategia de analisis:
1. Identificar un periodo COVID.
2. Marcarlo para analisis.
3. Excluirlo del entrenamiento del modelo.

In [ ]:
# Definimos periodo COVID (ajustable para clase)
covid_start = pd.Timestamp("2020-03-01")
covid_end = pd.Timestamp("2022-12-31")

df["is_covid_period"] = (df["date"] >= covid_start) & (df["date"] <= covid_end)

print("Filas en periodo COVID:", df["is_covid_period"].sum())
print("Filas fuera de COVID:", (~df["is_covid_period"]).sum())

In [ ]:
# Visualizamos para entender el impacto
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["y"], color="tab:blue", label="Serie completa")
plt.axvspan(covid_start, covid_end, color="red", alpha=0.15, label="Periodo COVID")
plt.title("Demanda diaria con periodo COVID resaltado")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend()
plt.show()

## 7) Serie para modelado (sin COVID) y test de estacionariedad

Creamos una serie de entrenamiento mas estable excluyendo COVID.

In [ ]:
df_model = df[~df["is_covid_period"]].copy()
df_model = df_model.set_index("date")

print("Rango modelado:", df_model.index.min(), "->", df_model.index.max())
print("Filas modelado:", len(df_model))

# Test ADF
adf_stat, adf_pvalue, *_ = adfuller(df_model["y"].dropna())
print("ADF statistic:", round(adf_stat, 4))
print("p-value:", round(adf_pvalue, 6))
print("Conclusion:", "Estacionaria" if adf_pvalue < 0.05 else "No estacionaria")

## 8) Forecast con baseline estacional semanal y SARIMAX + intervalos

Dividimos en train/test cronologico y comparamos dos modelos:
- Baseline estacional semanal: valor de hace 7 dias.
- SARIMAX con estacionalidad semanal.

In [ ]:
# Split temporal 80/20
split_idx = int(len(df_model) * 0.8)
train = df_model.iloc[:split_idx].copy()
test = df_model.iloc[split_idx:].copy()

print("Train:", train.index.min(), "->", train.index.max(), "|", len(train), "filas")
print("Test:", test.index.min(), "->", test.index.max(), "|", len(test), "filas")

In [ ]:
# Baseline estacional semanal (lag 7)
combined = pd.concat([train[["y"]], test[["y"]]])
seasonal_pred = combined["y"].shift(7).loc[test.index]

# Intervalo simple con std de residuo historico del baseline en train
train_resid = train["y"] - train["y"].shift(7)
sigma = train_resid.dropna().std()
seasonal_lower = seasonal_pred - 1.96 * sigma
seasonal_upper = seasonal_pred + 1.96 * sigma

In [ ]:
# SARIMAX sencillo con estacionalidad semanal (periodo 7)
sarimax = SARIMAX(
    train["y"],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_result = sarimax.fit(disp=False)

sarimax_forecast = sarimax_result.get_forecast(steps=len(test))
sarimax_pred = sarimax_forecast.predicted_mean
sarimax_ci = sarimax_forecast.conf_int(alpha=0.05)
sarimax_lower = sarimax_ci.iloc[:, 0]
sarimax_upper = sarimax_ci.iloc[:, 1]

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

metrics = pd.DataFrame(
    {
        "Modelo": ["Baseline semanal", "SARIMAX"],
        "MAE": [
            mean_absolute_error(test["y"], seasonal_pred),
            mean_absolute_error(test["y"], sarimax_pred),
        ],
        "RMSE": [
            rmse(test["y"], seasonal_pred),
            rmse(test["y"], sarimax_pred),
        ],
    }
)
metrics

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train.index, train["y"], label="Train", color="tab:blue", alpha=0.5)
plt.plot(test.index, test["y"], label="Test real", color="black")

plt.plot(test.index, seasonal_pred, label="Baseline semanal", color="tab:orange", linestyle="--")
plt.fill_between(test.index, seasonal_lower, seasonal_upper, color="tab:orange", alpha=0.2, label="IC 95% baseline")

plt.plot(test.index, sarimax_pred, label="SARIMAX", color="tab:green")
plt.fill_between(test.index, sarimax_lower, sarimax_upper, color="tab:green", alpha=0.2, label="IC 95% SARIMAX")

plt.title("Forecast con intervalos de confianza")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend(loc="best")
plt.show()

## 9) Backtesting walk-forward (horizonte 1 dia)

Usamos un bloque final de datos para simular prediccion diaria en condiciones reales.

In [ ]:
# Serie reducida para backtesting rapido en clase
bt_series = df_model["y"].iloc[-220:].copy()

real_values = []
pred_seasonal = []
pred_sarimax = []
dates = []

start = int(len(bt_series) * 0.7)

for i in range(start, len(bt_series)):
    history = bt_series.iloc[:i]
    y_real = bt_series.iloc[i]

    # Baseline semanal: valor 7 dias atras (si existe)
    if i >= 7:
        y_seasonal = bt_series.iloc[i - 7]
    else:
        y_seasonal = history.iloc[-1]

    # SARIMAX a 1 paso
    model_bt = SARIMAX(
        history,
        order=(1, 0, 1),
        seasonal_order=(1, 0, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    y_sarimax = model_bt.forecast(steps=1).iloc[0]

    real_values.append(y_real)
    pred_seasonal.append(y_seasonal)
    pred_sarimax.append(y_sarimax)
    dates.append(bt_series.index[i])

bt_results = pd.DataFrame(
    {
        "date": dates,
        "y_real": real_values,
        "y_baseline": pred_seasonal,
        "y_sarimax": pred_sarimax,
    }
).set_index("date")

print("MAE baseline semanal:", round(mean_absolute_error(bt_results["y_real"], bt_results["y_baseline"]), 2))
print("MAE SARIMAX:", round(mean_absolute_error(bt_results["y_real"], bt_results["y_sarimax"]), 2))

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(bt_results.index, bt_results["y_real"], label="Real", color="black")
plt.plot(bt_results.index, bt_results["y_baseline"], label="Baseline semanal", linestyle="--", alpha=0.8)
plt.plot(bt_results.index, bt_results["y_sarimax"], label="SARIMAX", alpha=0.8)
plt.title("Backtesting walk-forward")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend()
plt.show()

## 10) Conclusiones

- El dominio (calendario y eventos) es clave en series reales.
- El heatmap ayuda a ver patrones estacionales de un vistazo.
- Excluir periodos anomalos (como COVID) puede mejorar aprendizaje del patron normal.
- Siempre comparar contra un baseline sencillo.
- Mostrar intervalos de confianza mejora la interpretacion del forecast.